# Key-Value Databases: A Practical Case Study
### When to choose Redis over PostgreSQL — and why it matters

---

In this notebook we explore **key-value stores** through hands-on examples, comparing them with relational databases.

We will use:
- **`fakeredis`** — a Redis-compatible in-memory store (no server needed)
- **`sqlite3`** — our relational database baseline

```bash
pip install fakeredis
```

In [28]:
pip install fakeredis

Note: you may need to restart the kernel to use updated packages.


## Part 0 — What IS a Key-Value Database?

A key-value (KV) store is the simplest possible database model:

```
  KEY              →   VALUE
──────────────────────────────────────────────────────
"user:42"          →   '{"name": "Alice", "age": 30}'
"session:abc"      →   "user:42"
"counter:hits"     →   1042
"cache:home_page"  →   "<html>...</html>"
```

Think of it as a **massive Python dictionary** that:
- Lives in memory (RAM) for ultra-fast access
- Is accessible over a network
- Can optionally persist to disk
- Is designed to scale across thousands of servers

**Famous KV stores:** Redis, Amazon DynamoDB, Apache Cassandra (partial), Riak, etcd, Memcached

In [29]:
import fakeredis
import sqlite3
import json
import time
import random
import timeit
from datetime import datetime, timedelta

# Fake Redis server (behaves like real Redis, no server needed)
r = fakeredis.FakeRedis(decode_responses=True)

print(" Setup complete — fakeredis and sqlite3 ready")

 Setup complete — fakeredis and sqlite3 ready


---
##  Part 1 — Core KV Operations

The basic vocabulary: `SET`, `GET`, `DEL`, `EXISTS`, `EXPIRE`.

In [30]:
r.set("greeting", "Hello, World!")

value = r.get("greeting")
print(f"GET greeting → {value}")

# Store complex objects by serializing to JSON
user = {"name": "Alice", "email": "alice@example.com", "age": 30}
r.set("user:42", json.dumps(user))

retrieved = json.loads(r.get("user:42"))
print(f"GET user:42  → {retrieved}")

GET greeting → Hello, World!
GET user:42  → {'name': 'Alice', 'email': 'alice@example.com', 'age': 30}


In [31]:
# EXISTS and delete ───────────────────────────────────────────────────────────
print(f"EXISTS user:42  > {bool(r.exists('user:42'))}")
print(f"EXISTS user:99  > {bool(r.exists('user:99'))}")

r.delete("greeting")
print(f"After DEL, GET greeting > {r.get('greeting')}  < returns None")

EXISTS user:42  > True
EXISTS user:99  > False
After DEL, GET greeting > None  < returns None


In [32]:
# ─── EXPIRE: automatic key expiration ─────────────────────────────────────────
# A superpower relational DBs DON'T have natively!

r.set("temp_token", "abc123", ex=6)  # expires in 6 seconds
print(f"Token TTL: {r.ttl('temp_token')} seconds remaining") # TTL --> Time To Live

time.sleep(2)
print(f"After 2s  > TTL: {r.ttl('temp_token')} seconds")

time.sleep(5)
print(f"After 7s  > token: {r.get('temp_token')}  < auto-deleted by Redis!")

Token TTL: 6 seconds remaining
After 2s  > TTL: 4 seconds
After 7s  > token: None  < auto-deleted by Redis!


---
## Part 2 — Rich Data Structures

Redis is not just strings. It has powerful built-in types that solve common problems elegantly.

In [33]:
# ─── HASHes: a dict inside a key ─────────────────────────────────────────────
# Perfect for representing entities without serializing the whole object

r.hset("user:100", mapping={
    "name": "Bob",
    "email": "bob@example.com",
    "score": "980"
})

print("Full user:", r.hgetall("user:100"))
print("Just name:", r.hget("user:100", "name"))

# Update a single field — without touching the rest, without loading the object
r.hset("user:100", "score", "1050")
print("Updated score:", r.hget("user:100", "score"))

Full user: {'name': 'Bob', 'email': 'bob@example.com', 'score': '980'}
Just name: Bob
Updated score: 1050


In [34]:
# ─── LISTs: ordered, push/pop from both ends ─────────────────────────────────
# Great for queues, activity feeds, recent items

r.delete("recent_views:user:42")
for product in ["laptop", "headphones", "keyboard", "monitor", "mouse"]:
    r.lpush("recent_views:user:42", product)  # newest first (left push)

# Get last 3 viewed items
recent = r.lrange("recent_views:user:42", 0, 2) 
# lrange: Redis command to get a range of elements from a list
# 0, 2 --> start index and end index (inclusive in Redis)
print("Last 3 viewed:", recent)

# Modify the list so that only items at positions 0 to 2 remain.
r.ltrim("recent_views:user:42", 0, 2)
# 0, -1 --> all items
print("After trim, list:", r.lrange("recent_views:user:42", 0, -1))

Last 3 viewed: ['mouse', 'monitor', 'keyboard']
After trim, list: ['mouse', 'monitor', 'keyboard']


In [35]:
# ─── SORTED SETs: ranked leaderboards ────────────────────────────────────────
# Elements have a score — automatically sorted, instantly queryable by rank

players = [
    ("alice", 9500), ("bob", 8200), ("carol", 11000),
    ("dave", 7800), ("eve", 10200)
]
for name, score in players:
    r.zadd("leaderboard", {name: score}) # zadd: Redis command to add members to a sorted set with their scores

# Top 3 players
top3 = r.zrevrange("leaderboard", 0, 2, withscores=True) # Give me the top 3 players from the leaderboard, from highest score to lowest, and include their scores.

print(" Top 3 players:")
for rank, (player, score) in enumerate(top3, 1):
    print(f"  #{rank}: {player:8s} — {int(score):,} pts")

 Top 3 players:
  #1: carol    — 11,000 pts
  #2: eve      — 10,200 pts
  #3: alice    — 9,500 pts


In [36]:
# ─── SETs: unique membership, fast set operations ────────────────────────────
# Who liked a post? Who follows both A and B?

r.sadd("likes:post:501", "alice", "bob", "carol", "dave")
r.sadd("likes:post:502", "alice", "eve", "carol")

print("Liked post 501:", sorted(r.smembers("likes:post:501")))
print("Liked BOTH posts (intersection):", sorted(r.sinter("likes:post:501", "likes:post:502")))
print("Liked EITHER post (union):", sorted(r.sunion("likes:post:501", "likes:post:502")))
print("Only liked 501 (difference):", sorted(r.sdiff("likes:post:501", "likes:post:502")))

Liked post 501: ['alice', 'bob', 'carol', 'dave']
Liked BOTH posts (intersection): ['alice', 'carol']
Liked EITHER post (union): ['alice', 'bob', 'carol', 'dave', 'eve']
Only liked 501 (difference): ['bob', 'dave']


---
##  Part 3 — Head-to-Head: KV vs Relational

Four real-world scenarios compared side-by-side.

### Case Study A —  Session Management

A web app needs to store user sessions. Sessions expire after 30 minutes of inactivity.

In [37]:
# ── RELATIONAL APPROACH ────────────────────────────────────────────────────────
conn = sqlite3.connect(":memory:")
cur = conn.cursor()
cur.execute("""
    CREATE TABLE sessions (
        session_id TEXT PRIMARY KEY,
        user_id    INTEGER,
        data       TEXT,
        expires_at TEXT
    )
""")

def sql_create_session(session_id, user_id, data):
    expires = (datetime.now() + timedelta(minutes=30)).isoformat()
    cur.execute(
        "INSERT INTO sessions VALUES (?, ?, ?, ?)",
        (session_id, user_id, json.dumps(data), expires)
    )
    conn.commit()

def sql_get_session(session_id):
    cur.execute(
        "SELECT data, expires_at FROM sessions WHERE session_id = ?",
        (session_id,)
    )
    row = cur.fetchone()
    if not row:
        return None
    data, expires_at = row
    #  Must manually check expiry on every read!
    if datetime.fromisoformat(expires_at) < datetime.now():
        cur.execute("DELETE FROM sessions WHERE session_id = ?", (session_id,))
        conn.commit()
        return None
    return json.loads(data)

def sql_cleanup_expired():
    #  Must run this as a cron job / background task!
    cur.execute("DELETE FROM sessions WHERE expires_at < ?", (datetime.now().isoformat(),))
    return cur.rowcount

sql_create_session("sess_abc", 42, {"cart": ["item1", "item2"], "theme": "dark"})
print("[SQL] Session:", sql_get_session("sess_abc"))
print("[SQL] Cleanup removed:", sql_cleanup_expired(), "expired rows")

[SQL] Session: {'cart': ['item1', 'item2'], 'theme': 'dark'}
[SQL] Cleanup removed: 0 expired rows


In [38]:
# ── KEY-VALUE APPROACH ────────────────────────────────────────────────────────
def kv_create_session(session_id, user_id, data):
    payload = {"user_id": user_id, **data} # Create a new dictionary with user_id, and then insert all key–value pairs from data into it.
    r.set(f"session:{session_id}", json.dumps(payload), ex=1800)  # 30 min TTL

def kv_get_session(session_id):
    val = r.get(f"session:{session_id}")
    return json.loads(val) if val else None

def kv_touch_session(session_id):
    """Reset TTL on user activity (sliding expiry)"""
    r.expire(f"session:{session_id}", 1800)

# No cleanup job needed — Redis handles expiry automatically!
kv_create_session("sess_xyz", 42, {"cart": ["item1", "item2"], "theme": "dark"})
print("[KV] Session:", kv_get_session("sess_xyz"))
print("[KV] TTL:", r.ttl("session:sess_xyz"), "seconds")
print("[KV] No cleanup job needed — Redis auto-expires! ")

[KV] Session: {'user_id': 42, 'cart': ['item1', 'item2'], 'theme': 'dark'}
[KV] TTL: 1800 seconds
[KV] No cleanup job needed — Redis auto-expires! 


### Case Study B —  Performance: Read Speed Benchmark

In [39]:
N = 5000

# Populate SQL
conn2 = sqlite3.connect(":memory:")
cur2 = conn2.cursor()
cur2.execute("CREATE TABLE products (id INTEGER PRIMARY KEY, name TEXT, price REAL, category TEXT)")
cur2.executemany(
    "INSERT INTO products VALUES (?, ?, ?, ?)",
    [(i, f"Product {i}", round(random.uniform(1, 500), 2), f"cat_{i % 10}") for i in range(N)]
)
conn2.commit()

# Populate Redis
r2 = fakeredis.FakeRedis(decode_responses=True)
pipe = r2.pipeline()
for i in range(N):
    pipe.hset(f"product:{i}", mapping={
        "name": f"Product {i}",
        "price": str(round(random.uniform(1, 500), 2)),
        "category": f"cat_{i % 10}"
    })
pipe.execute()

# Benchmark — same 500 random IDs for both
ids = [random.randint(0, N-1) for _ in range(500)]

def sql_reads():
    for id_ in ids:
        cur2.execute("SELECT name, price FROM products WHERE id = ?", (id_,))
        cur2.fetchone()

def kv_reads():
    for id_ in ids:
        r2.hgetall(f"product:{id_}")

sql_time = timeit.timeit(sql_reads, number=20)
kv_time  = timeit.timeit(kv_reads,  number=20)

print(f"SQL   — 500 reads × 20 runs: {sql_time*1000:.1f}ms total  ({sql_time/20*1000:.2f}ms/batch)")
print(f"Redis — 500 reads × 20 runs: {kv_time*1000:.1f}ms total  ({kv_time/20*1000:.2f}ms/batch)")
print()
ratio = sql_time / kv_time if kv_time < sql_time else kv_time / sql_time
winner = "Redis" if kv_time < sql_time else "SQLite"
print(f" {winner} is {ratio:.1f}x faster in this test")
print("\nNote: SQLite in-memory is highly optimized. In production with a real network,")
print("Redis wins dramatically due to its simpler data model and memory-first design.")

SQL   — 500 reads × 20 runs: 13.1ms total  (0.66ms/batch)
Redis — 500 reads × 20 runs: 364.9ms total  (18.25ms/batch)

 SQLite is 27.8x faster in this test

Note: SQLite in-memory is highly optimized. In production with a real network,
Redis wins dramatically due to its simpler data model and memory-first design.


### Case Study C —  Rate Limiting

Limit an API to N requests per user per minute. This is where KV databases truly shine.

In [40]:
# ── Rate limiting with Redis ───────────────────────────────────────────────────
RATE_LIMIT = 5   # max requests per window (5 for demo)
WINDOW_SEC = 60  # 1 minute

def check_rate_limit(user_id: str) -> dict:
    key = f"ratelimit:{user_id}:{int(time.time()) // WINDOW_SEC}"
    
    # INCR is atomic — no race conditions even under massive concurrency!
    count = r.incr(key)
    if count == 1:
        r.expire(key, WINDOW_SEC)  # set TTL on first request
    
    return {
        "allowed": count <= RATE_LIMIT,
        "count": count,
        "remaining": max(0, RATE_LIMIT - count),
        "reset_in": r.ttl(key)
    }

print("Simulating 7 API requests from user 'alice' (limit=5):\n")
for i in range(7):
    result = check_rate_limit("alice")
    status = " ALLOWED" if result["allowed"] else " BLOCKED"
    print(f"  Request #{i+1}: {status} | count={result['count']} | remaining={result['remaining']}")

Simulating 7 API requests from user 'alice' (limit=5):

  Request #1:  ALLOWED | count=1 | remaining=4
  Request #2:  ALLOWED | count=2 | remaining=3
  Request #3:  ALLOWED | count=3 | remaining=2
  Request #4:  ALLOWED | count=4 | remaining=1
  Request #5:  ALLOWED | count=5 | remaining=0
  Request #6:  BLOCKED | count=6 | remaining=0
  Request #7:  BLOCKED | count=7 | remaining=0


Why Redis INCR is perfect for rate limiting:

- Atomic — INCR is a single command, no race conditions possible
- Auto-expires — no cleanup job, no orphaned rows
- Millions of ops/sec — designed for this exact use case
- Cluster-ready — works across distributed Redis nodes

SQL-based rate limiting issues:

- Requires two queries (SELECT + UPDATE or INSERT) — not atomic
- Race conditions under high concurrency without explicit locks
- Manual cleanup for old windows
- Heavy write load on a transactional database
- Doesn't scale horizontally without distributed coordination


### Case Study D —  Shopping Cart

In [41]:
# ── RELATIONAL: normalized schema ─────────────────────────────────────────────
conn3 = sqlite3.connect(":memory:")
cur3 = conn3.cursor()
cur3.executescript("""
    CREATE TABLE carts (
        cart_id TEXT PRIMARY KEY,
        user_id INTEGER,
        created_at TEXT
    );
    CREATE TABLE cart_items (
        cart_id    TEXT,
        product_id INTEGER,
        quantity   INTEGER,
        unit_price REAL,
        PRIMARY KEY (cart_id, product_id)
    );
""")

def sql_add_to_cart(user_id, product_id, quantity, price):
    cart_id = f"cart_{user_id}"
    cur3.execute("INSERT OR IGNORE INTO carts VALUES (?, ?, ?)",
                 (cart_id, user_id, datetime.now().isoformat()))
    cur3.execute("""
        INSERT INTO cart_items VALUES (?, ?, ?, ?)
        ON CONFLICT(cart_id, product_id)
        DO UPDATE SET quantity = quantity + excluded.quantity
    """, (cart_id, product_id, quantity, price))
    conn3.commit()

def sql_get_cart(user_id):
    cur3.execute("""
        SELECT product_id, quantity, unit_price,
               quantity * unit_price AS subtotal
        FROM cart_items WHERE cart_id = ?
    """, (f"cart_{user_id}",))
    return cur3.fetchall()

sql_add_to_cart(1, 101, 2, 29.99)
sql_add_to_cart(1, 205, 1, 89.99)
sql_add_to_cart(1, 101, 1, 29.99)  # adds another unit

cart_sql = sql_get_cart(1)
total_sql = sum(row[3] for row in cart_sql)
print("[SQL] Cart:")
for row in cart_sql:
    print(f"  product_id={row[0]}, qty={row[1]}, price=${row[2]}, subtotal=${row[3]:.2f}")
print(f"[SQL] Total: ${total_sql:.2f}")

[SQL] Cart:
  product_id=101, qty=3, price=$29.99, subtotal=$89.97
  product_id=205, qty=1, price=$89.99, subtotal=$89.99
[SQL] Total: $179.96


In [42]:
# ── KEY-VALUE: HASH per user cart ─────────────────────────────────────────────
# One key per user, one hash field per product — no JOIN needed

def kv_add_to_cart(user_id, product_id, quantity, price):
    key = f"cart:{user_id}"
    existing = r.hget(key, str(product_id))
    if existing:
        item = json.loads(existing)
        item["qty"] += quantity
    else:
        item = {"qty": quantity, "price": price}
    r.hset(key, str(product_id), json.dumps(item))
    r.expire(key, 7 * 86400)  # abandoned carts expire in 7 days

def kv_get_cart(user_id):
    raw = r.hgetall(f"cart:{user_id}")
    items, total = [], 0
    for pid, val in raw.items():
        item = json.loads(val)
        subtotal = item["qty"] * item["price"]
        total += subtotal
        items.append({"product_id": pid, **item, "subtotal": subtotal})
    return items, total

kv_add_to_cart(1, 101, 2, 29.99)
kv_add_to_cart(1, 205, 1, 89.99)
kv_add_to_cart(1, 101, 1, 29.99)

items, total_kv = kv_get_cart(1)
print("[KV] Cart:")
for it in items:
    print(f"  product_id={it['product_id']}, qty={it['qty']}, price=${it['price']}, subtotal=${it['subtotal']:.2f}")
print(f"[KV] Total: ${total_kv:.2f}")
print(f"[KV] Cart auto-expires in {r.ttl('cart:1') // 86400} days (abandoned cart cleanup )")

[KV] Cart:
  product_id=101, qty=3, price=$29.99, subtotal=$89.97
  product_id=205, qty=1, price=$89.99, subtotal=$89.99
[KV] Total: $179.96
[KV] Cart auto-expires in 7 days (abandoned cart cleanup )


---
##  Part 4 — Architectural Patterns

How KV and relational databases work together in real systems.

In [43]:
# ── Cache-Aside Pattern ───────────────────────────────────────────────────────
# Redis sits in front of the DB to accelerate reads.
# Strategy: check cache first; on miss, load from DB and populate cache.

conn5 = sqlite3.connect(":memory:")
cur5 = conn5.cursor()
cur5.execute("CREATE TABLE products (id INTEGER PRIMARY KEY, name TEXT, price REAL, description TEXT)")
cur5.executemany("INSERT INTO products VALUES (?, ?, ?, ?)",
    [(i, f"Product {i}", round(random.uniform(5, 500), 2), "A great product. " * 20)
     for i in range(1000)])
conn5.commit()

cache = fakeredis.FakeRedis(decode_responses=True)
CACHE_TTL = 300  # 5 minutes
db_calls = [0]

def get_product(product_id: int):
    cache_key = f"product:{product_id}"
    
    cached = cache.get(cache_key)
    if cached:
        return json.loads(cached), "CACHE HIT"
    
    db_calls[0] += 1
    cur5.execute("SELECT id, name, price FROM products WHERE id = ?", (product_id,))
    row = cur5.fetchone()
    if not row:
        return None, "NOT FOUND"
    
    product = {"id": row[0], "name": row[1], "price": row[2]}
    cache.set(cache_key, json.dumps(product), ex=CACHE_TTL)
    return product, "CACHE MISS"

# Simulate realistic traffic — some products accessed many times
traffic = [42, 99, 42, 42, 7, 99, 42, 7, 200, 42, 99, 7]
print("Simulating 12 requests (products accessed repeatedly):\n")

for pid in traffic:
    product, status = get_product(pid)
    print(f"  Product #{pid:3d}: {status} — ${product['price']:.2f}")

print(f"\n  DB queries: {db_calls[0]} / {len(traffic)} requests")
print(f"  Cache hit rate: {(len(traffic) - db_calls[0]) / len(traffic) * 100:.0f}%")
print(f"  Saved {len(traffic) - db_calls[0]} expensive DB round-trips")

Simulating 12 requests (products accessed repeatedly):

  Product # 42: CACHE MISS — $491.47
  Product # 99: CACHE MISS — $476.19
  Product # 42: CACHE HIT — $491.47
  Product # 42: CACHE HIT — $491.47
  Product #  7: CACHE MISS — $13.65
  Product # 99: CACHE HIT — $476.19
  Product # 42: CACHE HIT — $491.47
  Product #  7: CACHE HIT — $13.65
  Product #200: CACHE MISS — $85.42
  Product # 42: CACHE HIT — $491.47
  Product # 99: CACHE HIT — $476.19
  Product #  7: CACHE HIT — $13.65

  DB queries: 4 / 12 requests
  Cache hit rate: 67%
  Saved 8 expensive DB round-trips


In [44]:
# ── Pub/Sub: Event-driven messaging between services ──────────────────────────
# Redis as a lightweight message bus

r_server = fakeredis.FakeServer()
pub_client = fakeredis.FakeRedis(server=r_server, decode_responses=True)
sub_client = fakeredis.FakeRedis(server=r_server, decode_responses=True)

received = []
pubsub = sub_client.pubsub()
pubsub.subscribe("orders:created", "orders:shipped")

# Order service publishes events
events = [
    ("orders:created", {"order_id": 1001, "user_id": 42, "total": 149.99}),
    ("orders:created", {"order_id": 1002, "user_id": 87, "total": 29.99}),
    ("orders:shipped", {"order_id": 1001, "tracking": "TRK-XYZ123"}),
]

for channel, payload in events:
    pub_client.publish(channel, json.dumps(payload))

# Notification service receives events
print(" Notification service receiving events from Redis Pub/Sub:\n")
for msg in pubsub.listen():
    if msg["type"] == "message":
        data = json.loads(msg["data"])
        received.append(data)
        print(f"  [{msg['channel']}] {data}")
    if len(received) >= len(events):
        break

print(f"\n {len(received)} events processed — services fully decoupled via Redis")

 Notification service receiving events from Redis Pub/Sub:

  [orders:created] {'order_id': 1001, 'user_id': 42, 'total': 149.99}
  [orders:created] {'order_id': 1002, 'user_id': 87, 'total': 29.99}
  [orders:shipped] {'order_id': 1001, 'tracking': 'TRK-XYZ123'}

 3 events processed — services fully decoupled via Redis


# ── Final Decision Framework ──────────────────────────────────────────────────

---
##  Part 6 — Key Design Patterns & Best Practices

In [45]:
# ── Key naming conventions ────────────────────────────────────────────────────
print("Key Naming Best Practices")
print("=" * 60)

examples = [
    ("", "user:42",                        "entity:id — simple and clear"),
    ("", "session:abc123",                 "type:value"),
    ("", "cache:product:42:v2",            "namespace:entity:id:version"),
    ("", "ratelimit:user:42:20250101_12",  "feature:entity:id:time_window"),
    ("", "leaderboard:game:chess",         "type:domain:scope"),
    ("", "42",                             "no namespace — collision risk!"),
    ("", "USER_DATA_FOR_ALICE_SMITH",      "verbose — use IDs, not names"),
    ("", "x",                             "too cryptic"),
    ("", "all_users",                     "storing all IDs in one key won't scale"),
]

for rating, key, note in examples:
    print(f"  {rating}  {key:42s}  # {note}")

Key Naming Best Practices
    user:42                                     # entity:id — simple and clear
    session:abc123                              # type:value
    cache:product:42:v2                         # namespace:entity:id:version
    ratelimit:user:42:20250101_12               # feature:entity:id:time_window
    leaderboard:game:chess                      # type:domain:scope
    42                                          # no namespace — collision risk!
    USER_DATA_FOR_ALICE_SMITH                   # verbose — use IDs, not names
    x                                           # too cryptic
    all_users                                   # storing all IDs in one key won't scale


In [46]:
# ── Pipelining: batch multiple commands in one round-trip ─────────────────────
r3 = fakeredis.FakeRedis(decode_responses=True)
N_OPS = 2000

# Without pipeline: N separate round-trips
start = time.perf_counter()
for i in range(N_OPS):
    r3.set(f"key:{i}", f"value:{i}")
no_pipe_ms = (time.perf_counter() - start) * 1000

# With pipeline: 1 round-trip
start = time.perf_counter()
pipe = r3.pipeline()
for i in range(N_OPS):
    pipe.set(f"key_p:{i}", f"value:{i}")
pipe.execute()
pipe_ms = (time.perf_counter() - start) * 1000

print(f"Without pipeline: {no_pipe_ms:.2f}ms for {N_OPS:,} ops")
print(f"With pipeline:    {pipe_ms:.2f}ms for {N_OPS:,} ops")
if pipe_ms < no_pipe_ms:
    print(f"\n⚡ Pipeline is {no_pipe_ms/pipe_ms:.1f}x faster")
print("\nIn production with real network latency (e.g. 1ms/call):")
print(f"  Without pipeline: ~{N_OPS}ms = {N_OPS/1000:.1f}s for {N_OPS:,} ops")
print(f"  With pipeline:    ~1ms  (all in one batch)")

Without pipeline: 95.49ms for 2,000 ops
With pipeline:    60.08ms for 2,000 ops

⚡ Pipeline is 1.6x faster

In production with real network latency (e.g. 1ms/call):
  Without pipeline: ~2000ms = 2.0s for 2,000 ops
  With pipeline:    ~1ms  (all in one batch)


In [47]:
# ── Atomic counters: INCR / INCRBY ────────────────────────────────────────────
r.delete("page_views:home")

# Simulate concurrent page views (INCR is atomic — no lost updates)
for _ in range(42):
    r.incr("page_views:home")

# Bulk increment
r.incrby("page_views:home", 100)

print(f"Page views: {r.get('page_views:home')}")
print("INCR is atomic — safe under concurrent writes, no transaction needed ")

Page views: 142
INCR is atomic — safe under concurrent writes, no transaction needed 


---
##  Summary

| # | Topic | Key Takeaway |
|---|-------|--------------|
| 1 | Core operations | SET, GET, DEL, EXPIRE — simple but very powerful |
| 2 | Rich data types | Hashes, Lists, Sets, Sorted Sets solve common patterns elegantly |
| 3 | Session management | Native TTL beats manual SQL expiry + cron jobs |
| 4 | Performance | O(1) lookups, no query planning overhead |
| 5 | Rate limiting | Atomic INCR is thread-safe and massively scalable |
| 6 | Shopping cart | Schema-free flexibility + auto-expiry for abandoned carts |
| 7 | When NOT to use | Complex JOINs, analytics, strong consistency → stick with SQL |
| 8 | Cache-aside | Redis + PostgreSQL = fast reads + reliable source of truth |
| 9 | Pub/Sub | Decouple microservices with Redis as a message bus |
| 10 | Best practices | Namespace your keys, use pipelining, leverage atomic ops |

### The Golden Rule

> **Use the right tool for the job.** Key-value stores excel at simple, high-frequency, schema-flexible operations with optional TTL. Relational databases excel at complex queries, relationships, and consistency guarantees. Most production systems use both.

### Further Reading
- [Redis Documentation](https://redis.io/docs/)
- [Amazon DynamoDB Best Practices](https://docs.aws.amazon.com/amazondynamodb/latest/developerguide/best-practices.html)
- [Martin Fowler — NoSQL Distilled](https://martinfowler.com/books/nosql.html)
- [Redis University (free courses)](https://university.redis.com/)